In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from scipy.stats import multivariate_normal
from scipy.spatial.transform import Rotation
from scipy.linalg import block_diag
import pinocchio as pin
import scipy.special as sp
import scipy.linalg as la
import time
import plotly.graph_objects as go

In [5]:

# ====================================================================
# #                                                                    #
# #                Lie Group Extended Kalman Filter.                   #
# #                       Elementary Functions                         #
# #                                                                    #
# ====================================================================


# ====================================================================
# A) Model Matrices Construction - Matrix F
# ====================================================================

def Fn_Calculator(Rnm1_nm1,am_nm1,abnm1_nm1,omega_nm1,xi1,xi2):

    # State transition matrix is 15 x 15
    F = np.zeros((15,15))

    # First Row
    F[0:3,0:3] = np.eye(3)
    F[0:3,3:6] = dt*np.eye(3)
    F[0:3,6:9] = -0.5*dt**2 * Rnm1_nm1 @ CartesianSpace2LieAlgebra_Wedge(am_nm1 - abnm1_nm1)
    F[0:3,9:12] = -0.5*Rnm1_nm1*dt**2

    # Second Row
    F[3:6,3:6] = np.eye(3)
    F[3:6,6:9] = -dt * Rnm1_nm1 @ CartesianSpace2LieAlgebra_Wedge(am_nm1 - abnm1_nm1)
    F[3:6,9:12] = -Rnm1_nm1*dt
    
    # Third Row
    F[6:9,6:9] = exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(omega_nm1*dt)).T
    F[6:9,12:15] = -dt*Left_Jacobian_Calculator(omega_nm1*dt)

    # Fourth Row
    F[9:12,9:12] = xi1 * np.eye(3)

    # Fifth Row
    F[12:15,12:15] = xi2 * np.eye(3)

    return F

# ====================================================================
# B) Model Matrices Construction - Matrix L_G
# ====================================================================

# Left Jacobian included in the structure of the Group
def LG_Calculator(dt, wn_hat):
    LG = np.zeros((15,15))
    I3 = np.eye(3)
    leftJ = Left_Jacobian_Calculator(wn_hat*dt)
    LG = block_diag(I3, I3, leftJ, I3, I3)
    return LG
    
# ====================================================================
# C) Model Matrices Construction - Matrix Qn
# ====================================================================

def Qn_Calculator(Rnm1_nm1,S_a,S_omega,S_ba,S_bw):
    Qn = np.zeros((15,15))

    # First Row of the Matrix
    Qn[0:3,0:3] = 0.25*(dt**4)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T
    Qn[0:3,3:6] = 0.5*(dt**3)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T

    # Second Row of the Matrix
    Qn[3:6,0:3] = 0.5*(dt**3)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T
    Qn[3:6,3:6] = (dt**2)*Rnm1_nm1 @ S_a @ Rnm1_nm1.T

    # Third Row of the Matrix
    Qn[6:9,6:9] = (dt**2)*S_omega

    # Fourth Row of the Matrix
    Qn[9:12,9:12] = S_ba

    # Fifth Row of the Matrix
    Qn[12:15,12:15] = S_bw

    return Qn

# ====================================================================
# D) Model Matrices Construction - Matrix H
# ====================================================================

def Hn_Calculator(Rn_nm1,L1,L2,L3,L4,pn_nm1):
    Hn = np.zeros((12,15))

    # First block of 3 rows
    Hn[0:3,0:3] = -Rn_nm1.T
    Hn[0:3,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L1 - pn_nm1))

    # Second block of 3 rows
    Hn[3:6,0:3] = -Rn_nm1.T
    Hn[3:6,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L2 - pn_nm1))

    # Third block of 3 rows
    Hn[6:9,0:3] = -Rn_nm1.T
    Hn[6:9,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L3 - pn_nm1))

    # Fourth block of 3 rows
    Hn[9:12,0:3] = -Rn_nm1.T
    Hn[9:12,6:9] = CartesianSpace2LieAlgebra_Wedge(Rn_nm1.T @ (L4 - pn_nm1))

    return Hn

# ====================================================================
# E) Construction of the Matrix that accumulates innovations from the 4 Landmarks
# ====================================================================

def Inovation_Calculator(y1, y2, y3, y4, Rn_nm1, L1, L2, L3, L4, pn_nm1):
    inov = np.zeros((12,1))
    inov[0:3,:]  = y1 - Rn_nm1.T @ (L1 - pn_nm1) 
    inov[3:6,:]  = y2 - Rn_nm1.T @ (L2 - pn_nm1) 
    inov[6:9,:]  = y3 - Rn_nm1.T @ (L3 - pn_nm1) 
    inov[9:12,:] = y4 - Rn_nm1.T @ (L4 - pn_nm1) 
    return inov

# ====================================================================
# F) Function to split the vector in R15 into 5 blocks in R3
# ====================================================================

def Split_Correction_Vector(vector_R15):
    a = vector_R15[0:3]
    b = vector_R15[3:6]
    c = vector_R15[6:9]
    d = vector_R15[9:12]
    e = vector_R15[12:15]
    return a,b,c,d,e

# ====================================================================
# G) Function to find the Kalman gain
# ====================================================================

def Kalman_Gain_Calculator(Pn_nm1,Hn,Rn):
    Sxx = Pn_nm1 @ Hn.T
    Syy = Hn @ Pn_nm1 @ Hn.T + Rn
    K = Sxx @ np.linalg.inv(Syy)
    return K

# ====================================================================
# H) Functions Regarding Lie Algebra/Group
# ====================================================================

# ====================================
# Wedge operator: R3 -> so(3) 
# ====================================
def CartesianSpace2LieAlgebra_Wedge(vector):
    # We receive the vector in R3
    x,y,z = vector.flatten()
    return np.array([[ 0,  -z,   y],
                     [ z,   0,  -x],
                     [-y,   x,   0]])

# ====================================
# Vee operator: so(3) -> R3 
# ====================================
def LieAlgebra2CartesianSpace_Vee(X):
    # We receive a skew symmetric Matrix in the Lie Algebra
    return np.array([-X[1,2],X[0,2],-X[0,1]]).reshape(3,1)

# ====================================
# exp map: so(3) -> SO(3)
# ====================================
def exp_map_from_so3_to_SO3(X):
    theta = np.sqrt(0.5 * np.trace(X.T @ X))

    if theta < 1e-4:
        # Taylor Series Approximation
        return np.eye(3) + X + (X @ X)/2
    else:
        # Rodrigues Formula for computing the matrix exponential of a skew-symmetric Matrix
        return np.eye(3) + (np.sin(theta) / theta) * X + ((1 - np.cos(theta)) / theta**2) * (X @ X)

# ====================================
# log map: SO(3) -> so(3)
# ====================================
def log_map_from_SO3_to_so3(X):
    # Force the outputs with clip function, between -1 and 1
    cos_theta = np.clip((np.trace(X) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if theta < 1e-4:
        # Taylor Series Approximation again
        return (X - X.T)/2
    else:
        return (0.5 * theta / np.sin(theta)) * (X - X.T)

# ====================================
# Left Jacobian Phi_SO3
# ====================================
def Left_Jacobian_Calculator(v_theta):
    # Note that v_theta belongs to R3
    norm = np.linalg.norm(v_theta)
    v_LieAlg = CartesianSpace2LieAlgebra_Wedge(v_theta)  
    
    if norm < 10**(-4):
        return np.eye(3) + 0.5*v_LieAlg + (1/6)*(v_LieAlg @ v_LieAlg)
    else:
        return np.eye(3) + ((1 - np.cos(norm))/norm**2)*v_LieAlg +\
        ((norm - np.sin(norm))/norm**3)*(v_LieAlg @ v_LieAlg)
        
# ====================================
# Adjoint Operator in SO(3)
# ====================================
def Adj_SO3(vector):
    I = np.eye(3)
    left = Left_Jacobian_Calculator(vector)
    return block_diag(I,I,left,I,I)
    
# ====================================================================
# I) ILS Generation - LOC and GS
# ====================================================================

# Generates an ILS estimate with sample_size from a Gaussian mixture
# Monte Carlo Approach
def Generator_ILS_Gaussian_Mixture(v_mean, M_cov, sample_size, glide_slope_incline):

    ILS_LOC = 0
    ILS_GS = 0
    
    for i in range(sample_size):
        z_sampled = np.random.multivariate_normal(v_mean.flatten(), M_cov)
                                                
        # Call the function to calculate ILS
        ILS_estimated = ILS_Calculator(z_sampled[0], z_sampled[1], z_sampled[2], glide_slope_incline)
        ILS_LOC = ILS_LOC + ILS_estimated[0]
        ILS_GS = ILS_GS + ILS_estimated[1]

    # Must divide by sample_size to obtain the mean value
    return [ILS_LOC/sample_size, ILS_GS/sample_size]
    

# ====================================================================
# J) ILS Calculator
# ====================================================================

def ILS_Calculator(x, y, z, glide_slope_incline):
    # Localizer
    LOC = y

    # Glide slope
    GS = np.rad2deg(np.atan(np.abs(z)/np.sqrt(x**2 + y**2))) - glide_slope_incline
    return [LOC, GS]

# ====================================================================
# K) Rotation Matrix from Euler Angles
# ====================================================================

def Rot_Euler_angles(vel_start):
    # Determine constant attitude (Pitch and Yaw) from velocity vector
    vx, vy, vz = vel_start
    yaw = np.arctan2(vy, vx)
    pitch = np.arctan2(-vz, np.sqrt(vx**2 + vy**2)) # -vz because Down is positive in NED
    roll = 0.0 

    # R_n is the rotation matrix from Body to Navigation (World) frame
    # Scipy Rotation uses scalar-last quaternions by default.
    rot = Rotation.from_euler('ZYX', [yaw, pitch, roll], degrees=False)
    return rot.as_matrix() 

print("================================")
print("Functions Loaded")
print("================================")


Functions Loaded


In [ ]:
# ====================================================================
# #                                                                    #
# #                Lie Group Extended Kalman Filter.                   #
# #                         Main Function                              #
# #                                                                    #
# ====================================================================

# We need a timer
start_time = time.perf_counter()

for i in range(1, N):

    # =========================================
    # Measurements available for each loop
    # =========================================
    # Measurements available at time i, note that yn_si has a 3x1 dimension
    yn_s1 = y_s1[:,i].reshape(3,1)
    yn_s2 = y_s2[:,i].reshape(3,1)
    yn_s3 = y_s3[:,i].reshape(3,1)
    yn_s4 = y_s4[:,i].reshape(3,1)
        
    # Accelerometer and gyroscope measurements
    amnm1 = acc_m[:,i - 1].reshape(3,1)
    wmnm1 = gyro_m[:,i - 1].reshape(3,1)

    # =========================================================
    # 1) EKF PREDICTION STEP
    # =========================================================
    # Divided into Mean and Covariance

    # ===================
    # Mean Prediction
    # ===================
    M_pn_n_est[i,:,:] = M_pn_n_est[i - 1,:,:] + M_vn_n_est[i - 1,:,:]*dt + \
                        (0.5*dt**2)*(g + M_Rn_n_est[i - 1] @ (amnm1 - M_bias_a_n_n[i - 1,:,:]))

    M_vn_n_est[i,:,:] = M_vn_n_est[i - 1,:,:] + \
                       dt*(g + M_Rn_n_est[i - 1] @ (amnm1 - M_bias_a_n_n[i - 1,:,:]))

    M_Rn_n_est[i,:,:] = M_Rn_n_est[i - 1,:,:] @ \
                exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(dt*(wmnm1 - M_bias_w_n_n[i - 1,:,:])))

    M_bias_a_n_n[i,:,:] = xi1*M_bias_a_n_n[i - 1,:,:]

    M_bias_w_n_n[i,:,:] = xi2*M_bias_w_n_n[i - 1,:,:]

    # Calculation of Matrices F, Lg, and Qm1
    F = Fn_Calculator(M_Rn_n_est[i - 1,:,:],amnm1,M_bias_a_n_n[i - 1,:,:],wmnm1 - M_bias_w_n_n[i - 1,:,:],xi1,xi2)
    LG = LG_Calculator(dt, wmnm1 - M_bias_w_n_n[i - 1,:,:])
    Qnm1 = Qn_Calculator(M_Rn_n_est[i - 1,:,:],Sa,Sw,Sba,Sbw)

    # ===================
    # Covariance Matrix Prediction
    # ===================
    M_Pnn[i,:,:] = F @ M_Pnn[i - 1,:,:] @ F.T + LG @ Qnm1 @ LG.T

    # =========================================================
    # 2) EKF UPDATE STEP
    # =========================================================
   
    # ===================
    # Mean Update
    # ===================
    # 1) Innovation calculation
    mu_n = Inovation_Calculator(yn_s1, yn_s2, yn_s3, yn_s4, M_Rn_n_est[i,:,:], L1, L2, L3, L4, M_pn_n_est[i,:,:])

    # 2) Observation Jacobian
    Hn = Hn_Calculator(M_Rn_n_est[i,:,:],L1,L2,L3,L4,M_pn_n_est[i,:,:])

    # 3) Kalman Gain
    K = Kalman_Gain_Calculator(M_Pnn[i,:,:],Hn,R_cal_n)

    # 4) Tangent plane perturbation
    mu_n_pert = K @ mu_n

    # 5) Vector separation
    delta_p,delta_v,delta_R,delta_ab,delta_wb = Split_Correction_Vector(mu_n_pert)

    # 6) Mean Update - The sum in R3 is the traditional sum of vectors, but in SO(3) we use the exp map
    M_pn_n_est[i,:,:] = M_pn_n_est[i,:,:] + delta_p
    M_vn_n_est[i,:,:] = M_vn_n_est[i,:,:] + delta_v
    
    # Note here that the perturbation is originally in R3, and we need to transport it to the Manifold
    M_Rn_n_est[i,:,:] = M_Rn_n_est[i,:,:] @ exp_map_from_so3_to_SO3(CartesianSpace2LieAlgebra_Wedge(delta_R))
    
    M_bias_a_n_n[i,:,:] = M_bias_a_n_n[i,:,:] + delta_ab
    M_bias_w_n_n[i,:,:] = M_bias_w_n_n[i,:,:] + delta_wb

    # Concatenated sn vector - ILS estimation we need all the variables in R15
    Rn_in_R3 = LieAlgebra2CartesianSpace_Vee(log_map_from_SO3_to_so3(M_Rn_n_est[i,:,:])).reshape(3,1)

    sn = np.concatenate([M_pn_n_est[i,:,:], M_vn_n_est[i,:,:], Rn_in_R3, M_bias_a_n_n[i,:,:], M_bias_w_n_n[i,:,:]])

    # ===================
    # Covariance Update (Joseph Form)
    # ===================
    # Term (I - K@H)
    ImKH = np.eye(15) - K @ Hn

    # Traditional Joseph structure (P_joseph = ImKH @ P @ ImKH.T + K @ R @ K.T)
    # We force the matrix to be positive definite
    P_joseph = ImKH @ M_Pnn[i,:,:] @ ImKH.T + K @ R_cal_n @ K.T

    # Application of geometric adjoints of the tangent plane at the boundaries
    adjoint = Adj_SO3(delta_R)
    M_Pnn[i,:,:] = adjoint @ P_joseph @ adjoint.T
    
    # ILS Generation
    LOC, GS = Generator_ILS_Gaussian_Mixture(sn, M_Pnn[i,:,:], sample_size, glide_slope_incline)
    M_ILS_LOC_err[i] = np.abs(LOC - ILS_LOC[i])
    M_ILS_GS_err[i]  = np.abs(GS - ILS_GSL[i])

# Time measurement to assess performance
end_time = time.perf_counter()

elapsed_time = end_time - start_time
elapsed_time_hours = elapsed_time / 3600

# Code for printing:
print("============================================")
print(f"End")
print(f"Time: {elapsed_time_hours:.6f} hours")
print("============================================")